In [1]:
!pip -q install ucimlrepo catboost

In [2]:
from ucimlrepo import fetch_ucirepo

In [3]:
adult = fetch_ucirepo(id=2)

X = adult.data.features.copy()
y = adult.data.targets.copy()

print("X shape:",X.shape)
print("y shape:",y.shape)
print("\nFeaturelar:")
print(X.columns.tolist())
print("\nTarget ustunlari:")
print(y.columns.tolist())

X shape: (48842, 14)
y shape: (48842, 1)

Featurelar:
['age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status', 'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss', 'hours-per-week', 'native-country']

Target ustunlari:
['income']


In [4]:
y.sample(5)

,income
31281,<=50K
607,<=50K
46772,<=50K.
16086,<=50K
1730,<=50K


In [5]:
y["income"].unique()

array(['<=50K', '>50K', '<=50K.', '>50K.'], dtype=object)

In [6]:
y_target = y["income"].map({
    '<=50K':0,
    '<=50K.':0,
    '<50K':1,
    '<50K.':1
})

In [7]:
y_target.value_counts()

,count
income,
0.0,37155


In [8]:
cat_cols_names = adult.variables.query("type == 'Categorical'")

In [9]:
adult.variables.query("type == 'Categorical'")

,name,role,type,demographic,description,units,missing_values
1,workclass,Feature,Categorical,Income,"Private, Self-emp-not-inc, Self-emp-inc, Feder...",None,yes
3,education,Feature,Categorical,Education Level,"Bachelors, Some-college, 11th, HS-grad, Prof-...",None,no
5,marital-status,Feature,Categorical,Other,"Married-civ-spouse, Divorced, Never-married, S...",None,no
6,occupation,Feature,Categorical,Other,"Tech-support, Craft-repair, Other-service, Sal...",None,yes
7,relationship,Feature,Categorical,Other,"Wife, Own-child, Husband, Not-in-family, Other...",None,no
8,race,Feature,Categorical,Race,"White, Asian-Pac-Islander, Amer-Indian-Eskimo,...",None,no
13,native-country,Feature,Categorical,Other,"United-States, Cambodia, England, Puerto-Rico,...",None,yes


In [10]:
cat_cols_names = adult.variables.query("type == 'Categorical'")["name"].to_list()
cat_cols_names.append("sex")
print(cat_cols_names)

['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'native-country', 'sex']


In [11]:
X[cat_cols_names] = X[cat_cols_names].astype("category")

In [12]:
X.dtypes

,0
age,int64
workclass,category
fnlwgt,int64
education,category
education-num,int64
marital-status,category
occupation,category
relationship,category
race,category
sex,category


In [13]:
y.isna().sum()

,0
income,0


In [14]:
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split

from sklearn.metrics import classification_report

In [15]:
X[cat_cols_names]=X[cat_cols_names].astype("str")

In [16]:
y_target = y["income"].map({
    '<=50K': 0,
    '<=50K.': 0,
    '>50K': 1,
    '>50K.': 1
})

X_train, X_test, y_train, y_test = train_test_split(X, y_target, test_size=0.2, stratify=y_target)

In [20]:
model = CatBoostClassifier(
    iterations=170,
    learning_rate=0.01,
    auto_class_weights="Balanced",
    loss_function="Logloss",
    cat_features= cat_cols_names,
    boosting_type='Ordered',
    task_type="GPU",
    devices="0"
)

In [21]:
model.fit(
    X_train, y_train,
    eval_set=(X_test, y_test),
    early_stopping_rounds=50,
    verbose=0
)

CatBoostClassifier(auto_class_weights='Balanced', boosting_type='Ordered', cat_features=['workclass', 'education', 'marital-status', 'occupation', 'relationship', 'race', 'native-country', 'sex'], devices='0', iterations=170, learning_rate=0.01, loss_function='Logloss', task_type='GPU')

In [22]:
print(classification_report(y_test,model.predict(X_test)))

              precision    recall  f1-score   support

           0       0.95      0.78      0.86      7431
           1       0.55      0.88      0.68      2338

    accuracy                           0.80      9769
   macro avg       0.75      0.83      0.77      9769
weighted avg       0.86      0.80      0.81      9769



In [24]:
print(classification_report(y_train, model.predict(X_train)))

              precision    recall  f1-score   support

           0       0.95      0.77      0.85     29724
           1       0.55      0.87      0.67      9349

    accuracy                           0.80     39073
   macro avg       0.75      0.82      0.76     39073
weighted avg       0.85      0.80      0.81     39073

